# 思路

1. 选用中证800股票池，构建以下因子：
   - 一个月动量
   - 三个月动量
   - 六个月动量
   - 十二个月动量
   - 波动率（20d std）
   - PB
   - ROE
   - (需要成交量数据) 流动性
2. （不做调整直接喂给lightgbm）
3. 横截面分位数 -> [0, 1]
4. 调整方向：因子值越高越好
5. 每月计算组合因子值，选择最高的20只股票
6. （预测回报大于设定值的概率，高于特点概率就交易）
7. 回归预测未来n天回报
   - X是因子值，y是预测值
   - 调参 
   - 回测，加手续费，加损失限制


# Multi-factor selection

In [1]:
import pickle
import pandas as pd
import numpy as np

def load_and_combine_all_factors():
    # fundamental factors
    try:
        with open('fundamental_factors.pkl', 'rb') as f:
            fundamental_factors = pickle.load(f)
        print(f"fundamental factors loaded: {list(fundamental_factors.keys())}")
    except FileNotFoundError:
        print("Fundamental factors not found!")
        fundamental_factors = {}
    
    # volume and price factors
    try:
        with open('volume_price_factors.pkl', 'rb') as f:
            volume_price_factors = pickle.load(f)
        print(f"volume and price factors loaded: {list(volume_price_factors.keys())}")
    except FileNotFoundError:
        print("Volume and price factors not found!")
        volume_price_factors = {}
    
    # Combine all factors
    all_factors = {**volume_price_factors, **fundamental_factors}
    
    print(f"\nTotal combined factors: {len(all_factors)}")
    print(f"All factor names: {list(all_factors.keys())}")
    
    return all_factors

all_factors = load_and_combine_all_factors()

fundamental factors loaded: ['PB', 'Leverage', 'ROE', 'ROA', 'YOY']
volume and price factors loaded: ['market_cap', 'log_market_cap', 'std_market_cap_sq', 'vol', 'cap_logcap_resid', 'vol_logcap_resid', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'Momentum_1M_Max', 'Momentum_12M_1M', 'Momentum_1M_vol_20d', 'cap_vol_elasticity', 'return_skewness', 'return_kurtosis', 'price_autocorr_5d', 'price_autocorr_10d', 'vol_acceleration']

Total combined factors: 24
All factor names: ['market_cap', 'log_market_cap', 'std_market_cap_sq', 'vol', 'cap_logcap_resid', 'vol_logcap_resid', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'Momentum_1M_Max', 'Momentum_12M_1M', 'Momentum_1M_vol_20d', 'cap_vol_elasticity', 'return_skewness', 'return_kurtosis', 'price_autocorr_5d', 'price_autocorr_10d', 'vol_acceleration', 'PB', 'Leverage', 'ROE', 'ROA', 'YOY']


In [11]:
# customize factor selection
factors_to_keep = ['Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'vol']

filtered_factors = {k: v for k, v in all_factors.items() if k in factors_to_keep}

print(f"Filtered factors: {len(filtered_factors)}")
print(f"Kept factors: {list(filtered_factors.keys())}")

all_factors = filtered_factors

Filtered factors: 5
Kept factors: ['vol', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M']


In [12]:
def create_factor_matrix(factors_dict): # ---时间没对齐？---
    """
    Convert factors dictionary to a single DataFrame with:
      - index = (date, stock)
      - columns = factors
    """
    factor_data = []
    
    for factor_name, factor_df in factors_dict.items():
        factor_long = factor_df.stack().reset_index()
        factor_long.columns = ['date', 'stock', factor_name]
        factor_long = factor_long.set_index(['date', 'stock'])
        factor_data.append(factor_long)

    combined_df = pd.concat(factor_data, axis=1)
    
    return combined_df

factor_df = create_factor_matrix(all_factors)

In [13]:
def rank_factors_cross_sectionally(factor_df, method='percentile'):
    """
    Rank factors cross-sectionally at each date
    method: 'percentile', 'rank', or 'zscore'
    """
    if method == 'percentile':
        # to percentile [0, 1]
        ranked_df = factor_df.groupby(level='date').rank(pct=True, ascending=False)
    elif method == 'rank':
        # rank: 1, 2, ...
        ranked_df = factor_df.groupby(level='date').rank(ascending=False)
    elif method == 'zscore':
        # normalization
        ranked_df = factor_df.groupby(level='date').apply(lambda x: (x - x.mean()) / x.std())
    
    return ranked_df

ranked_factors = rank_factors_cross_sectionally(factor_df, method='percentile')

In [14]:
def adjust_factor_directions(ranked_factors, pos_factors=None, neg_factors=None):
    """
    higher values -> better performance
    """
    adjusted_factors = ranked_factors.copy()
    
    # default direction
    if pos_factors is None:
        pos_factors = [
            "Momentum_1M", "Momentum_3M", "Momentum_12M", "Momentum_6M", "Momentum_12M_1M",
            "ROE", "ROA", "YOY",
            "Momentum_1M_Max", "Momentum_1M_vol_20d"
        ]
    
    if neg_factors is None:
        neg_factors = [
            "vol", "PB", "PE", "Leverage",
            "return_kurtosis", "return_skewness", "vol_acceleration",
            "price_autocorr_5d", "price_autocorr_10d"
        ] # lower is better
    
    # Check selected factors
    available_factors = list(ranked_factors.columns)
    
    existing_pos_factors = [f for f in pos_factors if f in available_factors]
    existing_neg_factors = [f for f in neg_factors if f in available_factors]
    unclassified_factors = [f for f in available_factors if f not in pos_factors + neg_factors]
    
    print(f"\nAvailable factors: {available_factors}")
    print(f"Existing positive factors: {existing_pos_factors}")
    print(f"Existing negative factors: {existing_neg_factors}")
    print(f"Unclassified factors: {unclassified_factors}")
    
    # lower is better -> higher is better
    for col in existing_neg_factors:
        adjusted_factors[col] = 1 - ranked_factors[col]
        print(f"Adjusted {col}")
    
    # unclassified
    # f unclassified_factors:
    #     print(f"\n Warning: Unclassified factors (assuming positive): {unclassified_factors}")
    #     print("   Please review and add them to pos_factors or neg_factors if needed")
    
    return adjusted_factors

adjusted_factors = adjust_factor_directions(
    ranked_factors,
    pos_factors=["Momentum_1M", "Momentum_3M", "Momentum_12M", "ROE", "ROA", "YOY"],
    neg_factors=["vol", "PB", "PE", "Leverage"]
)

# Verify adjustment
print("\n Factor ranges after adjustment:")
for col in adjusted_factors.columns:
    min_val = adjusted_factors[col].min()
    max_val = adjusted_factors[col].max()
    print(f"{col:20s}: [{min_val:.3f}, {max_val:.3f}]")


Available factors: ['vol', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M']
Existing positive factors: ['Momentum_1M', 'Momentum_3M', 'Momentum_12M']
Existing negative factors: ['vol']
Unclassified factors: ['Momentum_6M']
Adjusted vol

 Factor ranges after adjustment:
vol                 : [0.000, 1.000]
Momentum_1M         : [0.000, 1.000]
Momentum_3M         : [0.000, 1.000]
Momentum_6M         : [0.000, 1.000]
Momentum_12M        : [0.000, 1.000]


In [15]:
def create_category_weighted_composite(adjusted_factors, category_weights=None):
    """
    Weight factors by category and create a composite score.
    """
    factor_categories = {
        'momentum': ['Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M'],
        'value': ['PB', 'PE'],
        'quality': ['ROE', 'ROA', 'YOY'],
        'risk': ['vol', 'Leverage']
    }
    
    # customize category weights
    if category_weights is None:
        category_weights = {
            'momentum': 0.4,
            'value': 0.3,  
            'quality': 0.2,
            'risk': 0.1
        }
    
    print("Category weighting scheme:")
    for category, weight in category_weights.items():
        print(f" - {category}: {weight*100:.1f}%")
    
    available_factors = list(adjusted_factors.columns)
    composite_scores = []
    
    # Calculate weighted score for each category
    total_weight = 0
    composite_score = pd.Series(0, index=adjusted_factors.index)
    
    for category, factors in factor_categories.items():
        existing_factors = [f for f in factors if f in available_factors]
        
        if existing_factors and category in category_weights:
            # Average within category, then weight by category
            category_score = adjusted_factors[existing_factors].mean(axis=1)
            weight = category_weights[category]
            composite_score += category_score * weight
            total_weight += weight
            
            print(f"{category}: {len(existing_factors)} factors, weight {weight}")
        else:
            print(f"{category}: no factors available")
    
    # Normalize by total weight
    if total_weight > 0:
        composite_score = composite_score / total_weight
    
    print(f"\nCategory-weighted composite created (total weight: {total_weight})")
    print(f"Score range: [{composite_score.min():.3f}, {composite_score.max():.3f}]")
    
    return composite_score

# Create category-weighted composite
category_weighted_score = create_category_weighted_composite(
    adjusted_factors,
    category_weights={
        'momentum': 0.4,
        'value': 0.3, 
        'quality': 0.2,
        'risk': 0.1
    }
)

Category weighting scheme:
 - momentum: 40.0%
 - value: 30.0%
 - quality: 20.0%
 - risk: 10.0%
momentum: 4 factors, weight 0.4
value: no factors available
quality: no factors available
risk: 1 factors, weight 0.1

Category-weighted composite created (total weight: 0.5)
Score range: [0.003, 1.000]


# Light GBM

In [27]:
def prepare_factors_df_for_ml(adjusted_factors, composite_score):
    """
    Convert MultiIndex factor DataFrame to flat format for ML training
    """
    
    # Reset index to make date and stock regular columns
    factors_df = adjusted_factors.reset_index()
    
    # Add composite score
    composite_df = composite_score.reset_index()
    composite_df.columns = ['date', 'stock', 'composite']
    
    # Merge factors with composite score
    factors_df = factors_df.merge(composite_df, on=['date', 'stock'], how='left')
    
    print(f"Factors DataFrame for ML training:")
    print(f"  Shape: {factors_df.shape}")
    print(f"  Columns: {list(factors_df.columns)}")
    print(f"  Date range: {factors_df['date'].min()} to {factors_df['date'].max()}")
    print(f"  Number of unique stocks: {factors_df['stock'].nunique()}")
    
    return factors_df

# Create the factors_df
factors_df = prepare_factors_df_for_ml(adjusted_factors, category_weighted_score)

print("\nSample of factors_df:")
print(factors_df.head(10))

# Verify the structure
print(f"\nColumn verification:")
expected_cols = ['date', 'stock'] + list(adjusted_factors.columns) + ['composite']
actual_cols = list(factors_df.columns)
print(f"Expected: {expected_cols}")
print(f"Actual:   {actual_cols}")
print(f"Match: {expected_cols == actual_cols}")

Factors DataFrame for ML training:
  Shape: (10369426, 8)
  Columns: ['date', 'stock', 'vol', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'composite']
  Date range: 20100201 to 20231229
  Number of unique stocks: 3847

Sample of factors_df:
       date      stock       vol  Momentum_1M  Momentum_3M  Momentum_6M  \
0  20110114  000001.SZ  0.065151     0.222723     0.867866     0.952637   
1  20110114  000002.SZ  0.131530     0.052052     0.383033     0.526911   
2  20110114  000005.SZ  0.266134     0.241241     0.832391     0.939182   
3  20110114  000006.SZ  0.829133     0.107608     0.316195     0.292250   
4  20110114  000007.SZ  0.077443     0.212212     0.278149     0.433800   
5  20110114  000009.SZ  0.933620     0.801301     0.105398     0.106566   
6  20110114  000010.SZ  0.003073     0.145646     0.424936     0.836383   
7  20110114  000011.SZ  0.862938     0.093594     0.626221     0.910657   
8  20110114  000012.SZ  0.955132     0.974975     0.220051     0.10

In [29]:
def prepare_ml_training_data(factors_df, price_df, horizon=5, top_pct=0.20):
    """
    Complete data preparation for LightGBM training
    """
    
    print("=== Preparing ML Training Data ===")
    
    # Ensure date dtypes
    factors_df['date'] = pd.to_datetime(factors_df['date'], format="%Y%m%d")
    price_df['date'] = pd.to_datetime(price_df['date'], format="%Y%m%d")

    print(f"Factors data: {factors_df.shape}")
    print(f"Price data: {price_df.shape}")
    
    # 1. Merge factors with prices
    data = factors_df.merge(price_df[['date', 'stock', 'close']], 
                           on=['date', 'stock'], how='left')
    
    print(f"After price merge: {data.shape}")
    
    # 2. Create future return targets
    price_sorted = price_df.sort_values(['stock', 'date']).copy()
    price_sorted['future_close'] = price_sorted.groupby('stock')['close'].shift(-horizon)
    price_sorted['future_return'] = (price_sorted['future_close'] / price_sorted['close']) - 1
    
    targets = price_sorted[['date', 'stock', 'future_return']]
    data = data.merge(targets, on=['date', 'stock'], how='left')
    
    print(f"After target merge: {data.shape}")
    
    # 3. Remove rows without targets (last H days)
    data_clean = data.dropna(subset=['future_return']).reset_index(drop=True)
    print(f"After removing NaN targets: {data_clean.shape}")
    
    # 4. Mark top stocks by composite score for universe selection
    def mark_top_pct(df, score_col='composite', top_pct=top_pct):
        mask = df.groupby('date')[score_col].transform(
            lambda s: s >= s.quantile(1 - top_pct)
        )
        return mask
    
    data_clean['in_selection'] = mark_top_pct(data_clean, 'composite', top_pct)
    
    print(f"Selected stocks per date (avg): {data_clean.groupby('date')['in_selection'].sum().mean():.1f}")
    
    # 5. Define feature columns
    feature_cols = [col for col in data_clean.columns 
                   if col not in ['date', 'stock', 'close', 'future_return', 'in_selection']]
    
    print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")
    
    return data_clean, feature_cols

try:
    print("Loading price data from data/adjclose.csv...")
    
    price_df = pd.read_csv('data/adjclose.csv', index_col=0)
    price_df.index = pd.to_datetime(price_df.index, format="%Y%m%d")
    price_df.index.name = 'date'
    
    print(f"Price data loaded - Shape: {price_df.shape}")
    print(f"Date range: {price_df.index.min()} to {price_df.index.max()}")
    print(f"Stock columns: {list(price_df.columns[:5])}...")
    
    # Convert from wide format to long format
    price_df = price_df.stack().reset_index()
    price_df.columns = ['date', 'stock', 'close']
    
    # Sort and handle missing data with forward fill
    price_df = price_df.sort_values(['stock', 'date']).reset_index(drop=True)
    price_df['close'] = price_df.groupby('stock')['close'].fillna(method='ffill')
    price_df = price_df.dropna(subset=['close']).reset_index(drop=True)
    
    print(f"Price data processed - Final shape: {price_df.shape}")
    print(f"Sample of price data:")
    print(price_df.head())
except:
    print("No price data found.")

ml_data, feature_columns = prepare_ml_training_data(factors_df, price_df) # ml_data 时间问题

Loading price data from data/adjclose.csv...
Price data loaded - Shape: (3401, 5260)
Date range: 2010-01-04 00:00:00 to 2023-12-29 00:00:00
Stock columns: ['688191.SH', '001211.SZ', '688798.SH', '300071.SZ', '002557.SZ']...


C:\Users\17845\AppData\Local\Temp\ipykernel_49536\2776508037.py:71: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  price_df['close'] = price_df.groupby('stock')['close'].fillna(method='ffill')
C:\Users\17845\AppData\Local\Temp\ipykernel_49536\2776508037.py:71: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  price_df['close'] = price_df.groupby('stock')['close'].fillna(method='ffill')


Price data processed - Final shape: (11178730, 3)
Sample of price data:
        date      stock       close
0 2010-01-04  000001.SZ  851.320164
1 2010-01-05  000001.SZ  836.598896
2 2010-01-06  000001.SZ  822.236683
3 2010-01-07  000001.SZ  813.260300
4 2010-01-08  000001.SZ  811.465023
=== Preparing ML Training Data ===
Factors data: (10369426, 8)
Price data: (11178730, 3)
After price merge: (10369426, 9)
After target merge: (10369426, 10)
After removing NaN targets: (10350191, 10)
Selected stocks per date (avg): 561.2

Feature columns (6): ['vol', 'Momentum_1M', 'Momentum_3M', 'Momentum_6M', 'Momentum_12M', 'composite']


In [31]:
def setup_lightgbm_training(ml_data, feature_columns, train_start='2020-01-01', 
                           val_start='2022-01-01', test_start='2023-01-01'):
    """
    Set up LightGBM training with time-based splits
    """
    
    import lightgbm as lgb
    from sklearn.metrics import mean_squared_error
    
    print("=== Setting up LightGBM Training ===")
    
    # Convert dates
    train_start = pd.to_datetime(train_start)
    val_start = pd.to_datetime(val_start)  
    test_start = pd.to_datetime(test_start)
    
    # Time-based splits
    train_mask = (ml_data['date'] >= train_start) & (ml_data['date'] < val_start)
    val_mask = (ml_data['date'] >= val_start) & (ml_data['date'] < test_start)
    test_mask = ml_data['date'] >= test_start
    
    # filter by universe selection
    # train_mask = train_mask & ml_data['in_selection']
    # val_mask = val_mask & ml_data['in_selection']  
    # test_mask = test_mask & ml_data['in_selection']
    
    train_data = ml_data[train_mask].copy()
    val_data = ml_data[val_mask].copy()
    test_data = ml_data[test_mask].copy()
    
    print(f"Training set: {len(train_data):,} samples ({train_data['date'].min()} to {train_data['date'].max()})")
    print(f"Validation set: {len(val_data):,} samples ({val_data['date'].min()} to {val_data['date'].max()})")  
    print(f"Test set: {len(test_data):,} samples ({test_data['date'].min()} to {test_data['date'].max()})")
    
    X_train = train_data[feature_columns]
    y_train = train_data['future_return']
    
    X_val = val_data[feature_columns]
    y_val = val_data['future_return']
    
    X_test = test_data[feature_columns]
    y_test = test_data['future_return']
    
    # missing value fill by middle rank = 0.5
    X_train = X_train.fillna(0.5)
    X_val = X_val.fillna(0.5)
    X_test = X_test.fillna(0.5)
    
    print(f"X_train: {X_train.shape}")
    print(f"X_val: {X_val.shape}")  
    print(f"X_test: {X_test.shape}")
    
    # Create LightGBM datasets
    train_dataset = lgb.Dataset(X_train, label=y_train)
    val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)
    
    return {
        'train_dataset': train_dataset,
        'val_dataset': val_dataset, 
        'X_test': X_test,
        'y_test': y_test,
        'feature_columns': feature_columns,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data
    }

training_setup = setup_lightgbm_training(ml_data, feature_columns)

print(f"Features: {len(training_setup['feature_columns'])}")
print(f"Training samples: {len(training_setup['train_data']):,}")

=== Setting up LightGBM Training ===
Training set: 1,867,477 samples (2020-01-02 00:00:00 to 2021-12-31 00:00:00)
Validation set: 930,974 samples (2022-01-04 00:00:00 to 2022-12-30 00:00:00)
Test set: 911,739 samples (2023-01-03 00:00:00 to 2023-12-22 00:00:00)
X_train: (1867477, 6)
X_val: (930974, 6)
X_test: (911739, 6)
Features: 6
Training samples: 1,867,477


In [32]:
import lightgbm as lgb
def train_lightgbm_model(training_setup):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': 0
    }
    
    model = lgb.train(
        params,
        training_setup['train_dataset'],
        valid_sets=[training_setup['val_dataset']],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    
    return model

# Train the model
model = train_lightgbm_model(training_setup)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's rmse: 0.0707707


In [33]:
def simulate_trading_strategy(model, ml_data, feature_columns, 
                            start_date='2023-01-01', end_date='2024-12-31',
                            top_n_stocks=20, rebalance_freq='M', 
                            transaction_cost=0.001, initial_capital=1000000):
    """
    Simulate trading strategy using the trained model
    """
    
    print("=== Trading Simulation ===")
    
    # Filter data for simulation period
    sim_mask = (pd.to_datetime(ml_data['date']) >= pd.to_datetime(start_date)) & \
               (pd.to_datetime(ml_data['date']) <= pd.to_datetime(end_date))
    sim_data = ml_data[sim_mask].copy()
    sim_data['date'] = pd.to_datetime(sim_data['date'])
    
    print(f"Simulation period: {start_date} to {end_date}")
    print(f"Simulation data: {len(sim_data):,} records")
    
    # Generate predictions
    X_sim = sim_data[feature_columns].fillna(0.5)
    sim_data['predicted_return'] = model.predict(X_sim)
    
    # Create rebalancing dates
    if rebalance_freq == 'M':
        rebalance_dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    elif rebalance_freq == 'W':
        rebalance_dates = pd.date_range(start=start_date, end=end_date, freq='W-MON')
    else:
        rebalance_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
    # Portfolio tracking
    portfolio_history = []
    current_positions = {}
    cash = initial_capital
    
    for rebal_date in rebalance_dates:
        # Get data for rebalancing date
        rebal_data = sim_data[sim_data['date'] == rebal_date]
        
        if len(rebal_data) == 0:
            continue
            
        # Select top N stocks by predicted return
        top_stocks = rebal_data.nlargest(top_n_stocks, 'predicted_return')
        
        if len(top_stocks) == 0:
            continue
            
        # Equal weight allocation
        target_weight = 1.0 / len(top_stocks)
        target_positions = {}
        
        # Calculate target portfolio value
        total_portfolio_value = cash
        for stock, shares in current_positions.items():
            stock_data = rebal_data[rebal_data['stock'] == stock]
            if len(stock_data) > 0:
                current_price = stock_data['close'].iloc[0]
                total_portfolio_value += shares * current_price
        
        # Calculate target positions
        for _, row in top_stocks.iterrows():
            stock = row['stock']
            price = row['close']
            target_value = total_portfolio_value * target_weight
            target_shares = int(target_value / price)
            target_positions[stock] = target_shares
        
        # Execute trades
        trades = []
        
        # Sell positions not in new portfolio
        for stock, current_shares in current_positions.items():
            if stock not in target_positions:
                # Sell all shares
                stock_data = rebal_data[rebal_data['stock'] == stock]
                if len(stock_data) > 0:
                    price = stock_data['close'].iloc[0]
                    sell_value = current_shares * price
                    transaction_cost_amount = sell_value * transaction_cost
                    cash += sell_value - transaction_cost_amount
                    trades.append({
                        'date': rebal_date,
                        'stock': stock,
                        'action': 'sell',
                        'shares': current_shares,
                        'price': price,
                        'value': sell_value,
                        'cost': transaction_cost_amount
                    })
        
        # Adjust existing positions and buy new ones
        for stock, target_shares in target_positions.items():
            current_shares = current_positions.get(stock, 0)
            share_diff = target_shares - current_shares
            
            if share_diff != 0:
                price = top_stocks[top_stocks['stock'] == stock]['close'].iloc[0]
                
                if share_diff > 0:  # Buy
                    buy_value = share_diff * price
                    transaction_cost_amount = buy_value * transaction_cost
                    total_cost = buy_value + transaction_cost_amount
                    
                    if cash >= total_cost:
                        cash -= total_cost
                        trades.append({
                            'date': rebal_date,
                            'stock': stock,
                            'action': 'buy',
                            'shares': share_diff,
                            'price': price,
                            'value': buy_value,
                            'cost': transaction_cost_amount
                        })
                    else:
                        # Adjust shares if insufficient cash
                        affordable_shares = int((cash / (1 + transaction_cost)) / price)
                        if affordable_shares > 0:
                            target_shares = current_shares + affordable_shares
                            buy_value = affordable_shares * price
                            transaction_cost_amount = buy_value * transaction_cost
                            cash -= buy_value + transaction_cost_amount
                            trades.append({
                                'date': rebal_date,
                                'stock': stock,
                                'action': 'buy',
                                'shares': affordable_shares,
                                'price': price,
                                'value': buy_value,
                                'cost': transaction_cost_amount
                            })
                
                else:  # Sell partial
                    sell_shares = -share_diff
                    sell_value = sell_shares * price
                    transaction_cost_amount = sell_value * transaction_cost
                    cash += sell_value - transaction_cost_amount
                    trades.append({
                        'date': rebal_date,
                        'stock': stock,
                        'action': 'sell',
                        'shares': sell_shares,
                        'price': price,
                        'value': sell_value,
                        'cost': transaction_cost_amount
                    })
        
        # Update positions
        current_positions = target_positions.copy()
        
        # Calculate portfolio value
        portfolio_value = cash
        for stock, shares in current_positions.items():
            stock_data = rebal_data[rebal_data['stock'] == stock]
            if len(stock_data) > 0:
                price = stock_data['close'].iloc[0]
                portfolio_value += shares * price
        
        # Record portfolio state
        portfolio_history.append({
            'date': rebal_date,
            'portfolio_value': portfolio_value,
            'cash': cash,
            'num_positions': len(current_positions),
            'trades': len(trades)
        })
        
        print(f"{rebal_date.strftime('%Y-%m-%d')}: Portfolio Value: ${portfolio_value:,.0f}, "
              f"Cash: ${cash:,.0f}, Positions: {len(current_positions)}, Trades: {len(trades)}")
    
    return portfolio_history, trades

# Run the simulation
portfolio_history, trades = simulate_trading_strategy(
    model=model,
    ml_data=ml_data,
    feature_columns=feature_columns,
    start_date='2023-01-01',
    end_date='2024-12-31',
    top_n_stocks=20,
    rebalance_freq='M',  # Monthly rebalancing
    transaction_cost=0.001,  # 0.1% transaction cost
    initial_capital=1000000
)

=== Trading Simulation ===
Simulation period: 2023-01-01 to 2024-12-31
Simulation data: 911,739 records
2023-02-01: Portfolio Value: $999,001, Cash: $64, Positions: 20, Trades: 20
2023-03-01: Portfolio Value: $1,020,265, Cash: $10, Positions: 20, Trades: 39
2023-06-01: Portfolio Value: $1,084,566, Cash: $726, Positions: 20, Trades: 40
2023-08-01: Portfolio Value: $1,076,539, Cash: $25, Positions: 20, Trades: 40
2023-09-01: Portfolio Value: $1,011,704, Cash: $5,213, Positions: 20, Trades: 39
2023-11-01: Portfolio Value: $926,891, Cash: $9, Positions: 20, Trades: 40
2023-12-01: Portfolio Value: $1,024,544, Cash: $2, Positions: 20, Trades: 39


In [34]:
def analyze_trading_performance(portfolio_history, initial_capital=1000000):
    """
    Analyze trading strategy performance
    """
    
    portfolio_df = pd.DataFrame(portfolio_history)
    portfolio_df['date'] = pd.to_datetime(portfolio_df['date'])
    portfolio_df = portfolio_df.sort_values('date').reset_index(drop=True)
    
    # Calculate returns
    portfolio_df['total_return'] = (portfolio_df['portfolio_value'] / initial_capital) - 1
    portfolio_df['period_return'] = portfolio_df['portfolio_value'].pct_change()
    
    # Performance metrics
    total_return = portfolio_df['total_return'].iloc[-1]
    annualized_return = (1 + total_return) ** (365 / (portfolio_df['date'].iloc[-1] - portfolio_df['date'].iloc[0]).days) - 1
    volatility = portfolio_df['period_return'].std() * np.sqrt(12)  # Assuming monthly data
    sharpe_ratio = annualized_return / volatility if volatility > 0 else 0
    
    max_portfolio_value = portfolio_df['portfolio_value'].expanding().max()
    drawdown = (portfolio_df['portfolio_value'] - max_portfolio_value) / max_portfolio_value
    max_drawdown = drawdown.min()
    
    print("=== Performance Analysis ===")
    print(f"Initial Capital: ${initial_capital:,.0f}")
    print(f"Final Portfolio Value: ${portfolio_df['portfolio_value'].iloc[-1]:,.0f}")
    print(f"Total Return: {total_return:.2%}")
    print(f"Annualized Return: {annualized_return:.2%}")
    print(f"Volatility: {volatility:.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Maximum Drawdown: {max_drawdown:.2%}")
    
    return portfolio_df

# Analyze performance
performance_df = analyze_trading_performance(portfolio_history)

=== Performance Analysis ===
Initial Capital: $1,000,000
Final Portfolio Value: $1,024,544
Total Return: 2.45%
Annualized Return: 2.96%
Volatility: 24.98%
Sharpe Ratio: 0.12
Maximum Drawdown: -14.54%
